# Political Bias Detection - Exploratory Analysis

This notebook explores the dataset and validates feature engineering.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from preprocess import load_or_create_dataset
from features import PoliticalBiasFeatureExtractor

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Load Dataset

In [ ]:
# Load or create dataset
df = load_or_create_dataset()
print(f"Dataset shape: {df.shape}")
print(f"\nClass distribution:")
print(df['bias'].value_counts().sort_index())

# Show sample articles
print(f"\nSample articles:")
for idx in range(2):
    print(f"\n[{df['bias'].iloc[idx]}]")
    print(df['text'].iloc[idx][:200] + "...")

## 2. Feature Engineering Exploration

In [ ]:
# Initialize feature extractor
extractor = PoliticalBiasFeatureExtractor()

# Extract features for a subset
sample_texts = df['text'].tolist()[:100]
print("Extracting features from subset...")
features, hand_crafted = extractor.extract_all_features(sample_texts, fit_tfidf=True)

In [ ]:
# Analyze hand-crafted features
print("Hand-crafted features statistics:")
print(hand_crafted.describe())

In [ ]:
# Feature correlations
plt.figure(figsize=(12, 10))
correlation = hand_crafted.corr()
sns.heatmap(correlation, annot=False, cmap='coolwarm', center=0, 
            square=True, linewidths=0.5)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 3. Lexicon Feature Analysis

In [ ]:
# Extract lexicon features for all samples
lexicon_features = extractor.extract_lexicon_features(sample_texts)

# Group by bias category
df_subset = df.iloc[:100].reset_index(drop=True)
lexicon_features['bias'] = df_subset['bias']

print("Lexicon features by class:")
print(lexicon_features.groupby('bias')[['left_count', 'right_count', 'net_lean_score']].mean())

In [ ]:
# Visualize net lean scores
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot
lexicon_features.boxplot(column='net_lean_score', by='bias', ax=axes[0])
axes[0].set_title('Net Lean Score by Class')
axes[0].set_xlabel('Political Bias')
axes[0].set_ylabel('Net Lean Score')

# Bar plot of means
means = lexicon_features.groupby('bias')['net_lean_score'].mean()
means.plot(kind='bar', ax=axes[1], color='steelblue')
axes[1].set_title('Average Net Lean Score by Class')
axes[1].set_xlabel('Political Bias')
axes[1].set_ylabel('Average Net Lean Score')
axes[1].axhline(y=0, color='red', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## 4. Sentiment Analysis

In [ ]:
# Extract sentiment features
sentiment_features = extractor.extract_sentiment_features(sample_texts)
sentiment_features['bias'] = df_subset['bias']

print("Sentiment features by class:")
print(sentiment_features.groupby('bias')[['sentiment_pos', 'sentiment_neg', 'sentiment_compound']].mean())

In [ ]:
# Visualize sentiment distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, col in enumerate(['sentiment_pos', 'sentiment_neg', 'sentiment_compound']):
    sentiment_features.boxplot(column=col, by='bias', ax=axes[idx])
    axes[idx].set_title(f'{col.replace("_", " ").title()} by Class')
    axes[idx].set_xlabel('Political Bias')

plt.tight_layout()
plt.show()

## 5. Stylistic Features Analysis

In [ ]:
# Extract stylistic features
stylistic_features = extractor.extract_stylistic_features(sample_texts)
stylistic_features['bias'] = df_subset['bias']

print("Stylistic features by class:")
print(stylistic_features.groupby('bias')[['avg_sentence_length', 'avg_word_length', 
                                            'certainty_ratio', 'hedge_ratio']].mean())

In [ ]:
# Visualize certainty vs hedge
fig, ax = plt.subplots(figsize=(12, 6))

for bias_class in sorted(stylistic_features['bias'].unique()):
    data = stylistic_features[stylistic_features['bias'] == bias_class]
    ax.scatter(data['hedge_ratio'], data['certainty_ratio'], 
              label=bias_class, alpha=0.6, s=100)

ax.set_xlabel('Hedge Word Ratio')
ax.set_ylabel('Certainty Word Ratio')
ax.set_title('Certainty vs Hedging by Political Bias')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## 6. Overall Feature Statistics

In [ ]:
print(f"Total features extracted: {features.shape[1]}")
print(f"\nFeature breakdown:")
print(f"  - TF-IDF features: 5,000 (sparse)")
print(f"  - Lexicon features: 7 (dense)")
print(f"  - Sentiment features: 6 (dense)")
print(f"  - Stylistic features: 10 (dense)")
print(f"  - Entity features: 6 (dense)")
print(f"  - Readability features: 5 (dense)")
print(f"  - Total dense features: {features.shape[1] - 5000}")
print(f"\nFeature matrix shape: {features.shape}")
print(f"Feature matrix sparsity: {1.0 - (features.nnz / (features.shape[0] * features.shape[1])):.2%}")

## 7. Sample Texts by Bias Category

In [ ]:
print("Sample texts from each bias category:\n")
for bias_class in sorted(df['bias'].unique()):
    samples = df[df['bias'] == bias_class]['text'].head(1)
    print(f"\n{'='*60}")
    print(f"[{bias_class}]")
    print(f"{'='*60}")
    print(samples.values[0][[0]])